In [1]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings
import os
from pprint import pprint

참고 : 
- https://github.com/langchain-ai/langgraph/blob/main/examples/rag/langgraph_crag.ipynb


In [2]:
    # os.environ['OPENAI_API_KEY'] = '...'
os.environ['TAVILY_API_KEY'] = "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"

os.environ['LANGCHAIN_TRACING_V2'] = 'true'
os.environ['LANGCHAIN_ENDPOINT'] = 'https://api.smith.langchain.com'
    # os.environ['LANGCHAIN_API_KEY'] = '...'


## Data prep
- PDF 3개를 활용하여 chromaDB에 index의 형태로 저장

In [3]:
from langchain_community.document_loaders import PyPDFLoader

documents = list()

for pdf in ["papers/corrective rag.pdf", "papers/modular rag.pdf", "papers/self rag.pdf"]:
    loader = PyPDFLoader(pdf)
    pages = loader.load_and_split()
    documents.extend(pages)

In [4]:
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=250, chunk_overlap=0
)
doc_splits = text_splitter.split_documents(documents)

# Add to vectorDB
vectorstore = Chroma.from_documents(
    documents=doc_splits,
    collection_name="rag-chroma",
    embedding=OpenAIEmbeddings(),
)
retriever = vectorstore.as_retriever()

In [5]:
output = retriever.get_relevant_documents("What is modular rag?")

/Users/gieunkwak/opt/anaconda3/envs/rag_methods/lib/python3.11/site-packages/langchain_core/_api/deprecation.py:119: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 0.3.0. Use invoke instead.
  warn_deprecated(


In [7]:
pprint(output[1].page_content)

('5\n'
 'aligns the text more closely with data distribution through iter-\n'
 'ative self-enhancement [17], [18]. Routing in the RAG system\n'
 'navigates through diverse data sources, selecting the optimal\n'
 'pathway for a query, whether it involves summarization,\n'
 'specific database searches, or merging different information\n'
 'streams [19]. The Predict module aims to reduce redundancy\n'
 'and noise by generating context directly through the LLM,\n'
 'ensuring relevance and accuracy [13]. Lastly, the Task Adapter\n'
 'module tailors RAG to various downstream tasks, automating\n'
 'prompt retrieval for zero-shot inputs and creating task-specific\n'
 'retrievers through few-shot query generation [20], [21] .This\n'
 'comprehensive approach not only streamlines the retrieval pro-\n'
 'cess but also significantly improves the quality and relevance\n'
 'of the information retrieved, catering to a wide array of tasks\n'
 'and queries with enhanced precision and flexibility.\n'
 '2

## Set LLM

<span style="display: inline-block; background-color: #007acc; color: white; padding: 2px 6px; border-radius: 4px; margin: 2px;">
  1. Retrieval evaluator
</span>

<span style="display: inline-block; background-color: #28a745; color: white; padding: 2px 6px; border-radius: 4px; margin: 2px;">
  structured output
</span>

<span style="display: inline-block; background-color: #28a745; color: white; padding: 2px 6px; border-radius: 4px; margin: 2px;">
  langchain pipeline
</span>

In [13]:
question = "What is modular rag?"

In [9]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.pydantic_v1 import BaseModel, Field

# Data model
class GradeDocuments(BaseModel):
    """Binary score for relevance check on retrieved documents."""

    binary_score: str = Field(
        description="Documents are relevant to the question, 'yes' or 'no'"
    )

# LLM with function call
llm = ChatOpenAI(model="gpt-4o", temperature=0)
structured_llm_grader = llm.with_structured_output(GradeDocuments) # 정형화된 output을 강제함

# Prompt
system = """You are a grader assessing relevance of a retrieved document to a user question. \n 
    If the document contains keyword(s) or semantic meaning related to the question, grade it as relevant. \n
    Give a binary score 'yes' or 'no' score to indicate whether the document is relevant to the question."""
grade_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "Retrieved document: \n\n {document} \n\n User question: {question}"),
    ]
)

# langchain pipeline. 순차적으로 실행
retrieval_grader = grade_prompt | structured_llm_grader

In [10]:
# question = "Retrieval augmented generation"
docs = retriever.get_relevant_documents(question)
doc_txt = docs[0].page_content
print(retrieval_grader.invoke({"question": question, "document": doc_txt}))

binary_score='no'


In [14]:
question

'What is modular rag?'

In [11]:
doc_txt

'Zhang, Sandhini Agarwal, Katarina Slama, Alex Gray, John Schulman, Jacob Hilton, Fraser Kelton,\nLuke Miller, Maddie Simens, Amanda Askell, Peter Welinder, Paul Christiano, Jan Leike, and\n13'

<span style="display: inline-block; background-color: #007acc; color: white; padding: 2px 6px; border-radius: 4px; margin: 2px;">
  2. Knowledge refinement
</span>

<span style="display: inline-block; background-color: #28a745; color: white; padding: 2px 6px; border-radius: 4px; margin: 2px;">
  prompt
</span>

<span style="display: inline-block; background-color: #28a745; color: white; padding: 2px 6px; border-radius: 4px; margin: 2px;">
  langchain pipeline
</span>

In [15]:
# Data model
class RefineDocuments(BaseModel):
    """Knowledge refinement to strip unnecessary information."""

    refined_knowledge: str = Field(
        description="Information related to the question."
    )

# LLM with function call
llm = ChatOpenAI(model="gpt-4o", temperature=0)
structured_llm_refiner = llm.with_structured_output(RefineDocuments)

# Prompt
system = """You are an expert in extracting useful information from a passage.\n
            You should select sentences that are related to the user question.\n
            Bear in mind that you should not miss out any information."""
relevance_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "Retrieved document: \n\n {document} \n\n User question: {question}"),
    ]
)

# langchain pipeline. 순차적으로 실행
knowledge_strip = relevance_prompt | structured_llm_refiner

In [16]:
question = "Author of the paper"
docs = retriever.get_relevant_documents(question)
doc_txt = docs[1].page_content
output = knowledge_strip.invoke({"question": question, "document": doc_txt})

In [17]:
print(docs[1].page_content)

ICML 2020, 13-18 July 2020, Virtual Event , volume
119 of Proceedings of Machine Learning Research ,
pages 3929–3938. PMLR.
Gautier Izacard, Mathilde Caron, Lucas Hosseini,


In [18]:
print(output)

refined_knowledge='Gautier Izacard, Mathilde Caron, Lucas Hosseini'


<span style="display: inline-block; background-color: #007acc; color: white; padding: 2px 6px; border-radius: 4px; margin: 2px;">
  3. Output generator
</span>

<span style="display: inline-block; background-color: #28a745; color: white; padding: 2px 6px; border-radius: 4px; margin: 2px;">
  prompt
</span>

<span style="display: inline-block; background-color: #28a745; color: white; padding: 2px 6px; border-radius: 4px; margin: 2px;">
  langchain pipeline
</span>

In [19]:
### Generate
from langchain import hub
from langchain_core.output_parsers import StrOutputParser

# Prompt
prompt = hub.pull("rlm/rag-prompt")
# https://smith.langchain.com/hub/rlm/rag-prompt?organizationId=d7027af0-13d4-50bc-ab87-b575a5e80839

# LLM
llm = ChatOpenAI(model_name="gpt-4o", temperature=0)

# Post-processing
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Chain
rag_chain = prompt | llm | StrOutputParser()

# Run
generation = rag_chain.invoke({"context": docs, "question": question})
print(generation)

The authors of the paper are Zhang, Sandhini Agarwal, Katarina Slama, Alex Gray, John Schulman, Jacob Hilton, Fraser Kelton, Luke Miller, Maddie Simens, Amanda Askell, Peter Welinder, Paul Christiano, and Jan Leike.


In [ ]:
prompt

<span style="display: inline-block; background-color: #007acc; color: white; padding: 2px 6px; border-radius: 4px; margin: 2px;">
  4. Web search query writer
</span>

<span style="display: inline-block; background-color: #28a745; color: white; padding: 2px 6px; border-radius: 4px; margin: 2px;">
  prompt
</span>

<span style="display: inline-block; background-color: #28a745; color: white; padding: 2px 6px; border-radius: 4px; margin: 2px;">
  langchain pipeline
</span>

In [21]:
question

'Author of the paper'

In [20]:
### Question Re-writer

# LLM
llm = ChatOpenAI(model="gpt-4o", temperature=0)

# Prompt
system = """You a question re-writer that converts an input question to a better version that is optimized \n 
     for web search. Look at the input and try to reason about the underlying semantic intent / meaning."""
re_write_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        (
            "human",
            "Here is the initial question: \n\n {question} \n Formulate an improved question.",
        ),
    ]
)

question_rewriter = re_write_prompt | llm | StrOutputParser()
question_rewriter.invoke({"question": question})

'Who is the author of the paper?'

<span style="display: inline-block; background-color: #007acc; color: white; padding: 2px 6px; border-radius: 4px; margin: 2px;">
  4. Web search tool
</span>

<span style="display: inline-block; background-color: #28a745; color: white; padding: 2px 6px; border-radius: 4px; margin: 2px;">
  Tavily
</span>

In [22]:
### Search
from langchain_community.tools.tavily_search import TavilySearchResults

web_search_tool = TavilySearchResults(k=3)

## Create langGraph

In [23]:
from typing_extensions import TypedDict
from typing import List

# 현재 state를 설정
# 각 state 변수의 type 설정
class GraphState(TypedDict):
    """
    Represents the state of our graph.

    Attributes:
        question: question
        generation: LLM generation
        web_search: whether to add search
        documents: list of documents
    """

    question: str
    generation: str
    web_search: str
    documents: List[str]


<span style="display: inline-block; background-color: #007acc; color: white; padding: 2px 6px; border-radius: 4px; margin: 2px;">
  1. Graph nodes
</span>

<span style="display: inline-block; background-color: #28a745; color: white; padding: 2px 6px; border-radius: 4px; margin: 2px;">
  retrieve
</span>

<span style="display: inline-block; background-color: #28a745; color: white; padding: 2px 6px; border-radius: 4px; margin: 2px;">
  generate
</span>

<span style="display: inline-block; background-color: #28a745; color: white; padding: 2px 6px; border-radius: 4px; margin: 2px;">
  grade doc
</span>

<span style="display: inline-block; background-color: #28a745; color: white; padding: 2px 6px; border-radius: 4px; margin: 2px;">
  refine doc
</span>

<span style="display: inline-block; background-color: #28a745; color: white; padding: 2px 6px; border-radius: 4px; margin: 2px;">
  rewrite query for web search
</span>

<span style="display: inline-block; background-color: #28a745; color: white; padding: 2px 6px; border-radius: 4px; margin: 2px;">
  web search
</span>

In [24]:
from langchain.schema import Document


def retrieve(state):
    """
    Retrieve documents

    Args:
        state (dict): The current graph state

    Returns:
        state (dict): New key added to state, documents, that contains retrieved documents
    """
    print("---RETRIEVE---")
    question = state["question"]

    # Retrieval
    documents = retriever.get_relevant_documents(question)
    return {"documents": documents, "question": question}


def generate(state):
    """
    Generate answer

    Args:
        state (dict): The current graph state

    Returns:
        state (dict): New key added to state, generation, that contains LLM generation
    """
    print("---GENERATE---")
    question = state["question"]
    documents = state["documents"]

    # RAG generation
    generation = rag_chain.invoke({"context": documents, "question": question})
    return {"documents": documents, "question": question, "generation": generation}


def grade_documents(state):
    """
    Determines whether the retrieved documents are relevant to the question.

    Args:
        state (dict): The current graph state

    Returns:
        state (dict): Updates documents key with only filtered relevant documents
    """

    print("---CHECK DOCUMENT RELEVANCE TO QUESTION---")
    question = state["question"]
    documents = state["documents"]

    # Score each doc
    filtered_docs = []
    # web_search = "No"
    for d in documents:
        score = retrieval_grader.invoke(
            {"question": question, "document": d.page_content}
        )
        grade = score.binary_score
        if grade == "yes":
            print("---GRADE: DOCUMENT RELEVANT---", d.metadata)
            filtered_docs.append(d)
        else:
            print("---GRADE: DOCUMENT NOT RELEVANT---", d.metadata)
            # web_search = "Yes"
            continue
    if len(filtered_docs)==0:
        web_search = "Yes"
    else:
        web_search = "No"
    return {"documents": filtered_docs, "question": question, "web_search": web_search}


def refine_documents(state):
    """
    Refine and strip documents

    Args:
        state (dict): The current graph state

    Returns:
        state (dict): Updates documents, that contains refined documents
    """
    print("---REFINE KNOWLEDGE---")
    question = state["question"]
    documents = state["documents"]

    refined_docs = []

    # Refinement
    for d in documents:
        refined_d = knowledge_strip.invoke({"question": question, "document": d})
        refined_docs.append(refined_d)

    return {"documents": refined_docs, "question": question, "web_search": web_search}


def transform_query(state):
    """
    Transform the query to produce a better question.

    Args:
        state (dict): The current graph state

    Returns:
        state (dict): Updates question key with a re-phrased question
    """

    print("---TRANSFORM QUERY---")
    question = state["question"]
    documents = state["documents"]

    # Re-write question
    better_question = question_rewriter.invoke({"question": question})
    return {"documents": documents, "question": better_question}


def web_search(state):
    """
    Web search based on the re-phrased question.

    Args:
        state (dict): The current graph state

    Returns:
        state (dict): Updates documents key with appended web results
    """

    print("---WEB SEARCH---")
    question = state["question"]
    documents = state["documents"]

    # Web search
    docs = web_search_tool.invoke({"query": question})
    web_results = "\n".join([d["content"] for d in docs])
    web_results = Document(page_content=web_results)
    documents.append(web_results)

    return {"documents": documents, "question": question}

<span style="display: inline-block; background-color: #007acc; color: white; padding: 2px 6px; border-radius: 4px; margin: 2px;">
  2. Graph edges
</span>

<span style="display: inline-block; background-color: #28a745; color: white; padding: 2px 6px; border-radius: 4px; margin: 2px;">
  retrieve
</span>

In [25]:
### Edges
def decide_to_generate(state):
    """
    Determines whether to generate an answer, or re-generate a question.

    Args:
        state (dict): The current graph state

    Returns:
        str: Binary decision for next node to call
    """

    print("---ASSESS GRADED DOCUMENTS---")
    web_search = state["web_search"]

    if web_search == "Yes":
        # All documents have been filtered check_relevance
        # We will re-generate a new query
        print(
            "---DECISION: ALL DOCUMENTS ARE NOT RELEVANT TO QUESTION, TRANSFORM QUERY---"
        )
        return "transform_query"
    else:
        # We have relevant documents, so generate answer
        print("---DECISION: USE RETRIEVED---")
        return "refine_documents"

## Build graph

In [26]:
from langgraph.graph import END, StateGraph

workflow = StateGraph(GraphState)

# Define the nodes
workflow.add_node("retrieve", retrieve)  # retrieve
workflow.add_node("grade_documents", grade_documents)  # grade documents
workflow.add_node("refine_documents", refine_documents)  # refine documents
workflow.add_node("generate", generate)  # generatae
workflow.add_node("transform_query", transform_query)  # transform_query
workflow.add_node("web_search_node", web_search)  # web search

# Build graph
workflow.set_entry_point("retrieve")
workflow.add_edge("retrieve", "grade_documents")
workflow.add_conditional_edges(
    "grade_documents",
    decide_to_generate,
    {
        "transform_query": "transform_query",
        "refine_documents": "refine_documents",
    },
)
workflow.add_edge("transform_query", "web_search_node")
workflow.add_edge("web_search_node", "refine_documents")
workflow.add_edge("refine_documents", "generate")
workflow.add_edge("generate", END)

# Compile
app = workflow.compile()

## Test

In [27]:
from pprint import pprint

# Run
inputs = {"question": "What are the main advantages if we combine self rag and corrective rag?"}
for output in app.stream(inputs):
    for key, value in output.items():
        # Node
        pprint(f"Node '{key}':")
        # Optional: print full state at each node
        # pprint.pprint(value["keys"], indent=2, width=80, depth=None)
    pprint("\n---\n")

# Final generation
pprint(value["generation"])

---RETRIEVE---
"Node 'retrieve':"
'\n---\n'
---CHECK DOCUMENT RELEVANCE TO QUESTION---
---GRADE: DOCUMENT RELEVANT--- {'page': 7, 'source': 'papers/corrective rag.pdf'}
---GRADE: DOCUMENT NOT RELEVANT--- {'page': 2, 'source': 'papers/self rag.pdf'}
---GRADE: DOCUMENT NOT RELEVANT--- {'page': 8, 'source': 'papers/corrective rag.pdf'}
---GRADE: DOCUMENT NOT RELEVANT--- {'page': 2, 'source': 'papers/self rag.pdf'}
---ASSESS GRADED DOCUMENTS---
---DECISION: USE RETRIEVED---
"Node 'grade_documents':"
'\n---\n'
---REFINE KNOWLEDGE---
"Node 'refine_documents':"
'\n---\n'
---GENERATE---
"Node 'generate':"
'\n---\n'
('Combining self-RAG and corrective-RAG (CRAG) enhances the robustness of '
 "generation to retrieval performance drops. CRAG's lightweight retrieval "
 'evaluator and optimized knowledge utilization improve automatic '
 'self-correction and efficient use of retrieved documents. This combination '
 'results in superior adaptability and generalizability in RAG-based '
 'approaches.')

In [28]:
from pprint import pprint

# Run
inputs = {"question": "What is the main difference of modular rag and advanced rag?"}
for output in app.stream(inputs):
    for key, value in output.items():
        # Node
        pprint(f"Node '{key}':")
        # Optional: print full state at each node
        # pprint.pprint(value["keys"], indent=2, width=80, depth=None)
    pprint("\n---\n")

# Final generation
pprint(value["generation"])

---RETRIEVE---
"Node 'retrieve':"
'\n---\n'
---CHECK DOCUMENT RELEVANCE TO QUESTION---
---GRADE: DOCUMENT RELEVANT--- {'page': 3, 'source': 'papers/modular rag.pdf'}
---GRADE: DOCUMENT RELEVANT--- {'page': 3, 'source': 'papers/modular rag.pdf'}
---GRADE: DOCUMENT RELEVANT--- {'page': 4, 'source': 'papers/modular rag.pdf'}
---GRADE: DOCUMENT NOT RELEVANT--- {'page': 6, 'source': 'papers/modular rag.pdf'}
---ASSESS GRADED DOCUMENTS---
---DECISION: USE RETRIEVED---
"Node 'grade_documents':"
'\n---\n'
---REFINE KNOWLEDGE---
"Node 'refine_documents':"
'\n---\n'
---GENERATE---
"Node 'generate':"
'\n---\n'
('The main difference between Modular RAG and Advanced RAG is that Modular RAG '
 'introduces specialized components and allows for module substitution or '
 'reconfiguration, offering greater flexibility and adaptability. Advanced '
 'RAG, while optimized, still follows a more fixed, chain-like structure. '
 'Modular RAG supports both sequential processing and integrated end-to-end '
 'tra

---

In [29]:
from pprint import pprint

# Run
inputs = {"question": "How old is Jack Sparrow?"}
for output in app.stream(inputs):
    for key, value in output.items():
        # Node
        pprint(f"Node '{key}':")
        # Optional: print full state at each node
        # pprint.pprint(value["keys"], indent=2, width=80, depth=None)
    pprint("\n---\n")

# Final generation
pprint(value["generation"])

---RETRIEVE---
"Node 'retrieve':"
'\n---\n'
---CHECK DOCUMENT RELEVANCE TO QUESTION---
---GRADE: DOCUMENT NOT RELEVANT--- {'page': 4, 'source': 'papers/self rag.pdf'}
---GRADE: DOCUMENT NOT RELEVANT--- {'page': 23, 'source': 'papers/self rag.pdf'}
---GRADE: DOCUMENT NOT RELEVANT--- {'page': 12, 'source': 'papers/self rag.pdf'}
---GRADE: DOCUMENT NOT RELEVANT--- {'page': 11, 'source': 'papers/corrective rag.pdf'}
---ASSESS GRADED DOCUMENTS---
---DECISION: ALL DOCUMENTS ARE NOT RELEVANT TO QUESTION, TRANSFORM QUERY---
"Node 'grade_documents':"
'\n---\n'
---TRANSFORM QUERY---
"Node 'transform_query':"
'\n---\n'
---WEB SEARCH---
"Node 'web_search_node':"
'\n---\n'
---REFINE KNOWLEDGE---
"Node 'refine_documents':"
'\n---\n'
---GENERATE---
"Node 'generate':"
'\n---\n'
'Jack Sparrow is between 38 and 39 years old in "The Curse of the Black Pearl."'
